<a href="https://colab.research.google.com/github/Rrricardi/reabastecimiento-inteligente-de-inventario-ml/blob/main/Create_data_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Asegurar que el directorio 'data/' exista
os.makedirs("data", exist_ok=True)

# Creamos una base de datos SQLite en data/
engine = create_engine("sqlite:///data/retail_store.db")


def build_database():
  print("1/4 Cargando CSVs de Kaggle...")
  df_train = pd.read_csv("/content/drive/MyDrive/Data/train.csv")
  df_stores = pd.read_csv("/content/drive/MyDrive/Data/stores.csv")
  df_oil = pd.read_csv("/content/drive/MyDrive/Data/oil.csv")

  # Formatear fechas
  df_train["date"] = pd.to_datetime(df_train["date"])
  df_oil["date"] = pd.to_datetime(df_oil["date"])

  print("2/4 Creando tabla de dimensiones de productos/familias...")
  # Kaggle agrupa por 'family' (33 familias de productos)
  families = df_train["family"].unique()
  np.random.seed(42)

  # Parametros simulados de inventario (costo, margen y lead time en dias)
  df_products = pd.DataFrame({
      "product_id": families,
      "product_name": families,
      "category": "Retail",
      "unit_cost": np.random.uniform(2.5, 30.0, size=len(families)).round(2),
      "lead_time_days": np.random.choice([3, 5, 7, 10], size=len(families)),
  })
  df_products["selling_price"] = (df_products["unit_cost"] * 1.30).round(2)

  print("3/4 Preparando tabla de ventas y petroleo...")
  # Rellenar valores nulos de petroleo mediante interpolacion lineal hacia adelante
  df_oil["dcoilwtico"] = df_oil["dcoilwtico"].ffill().bfill()

  print("4/4 Guardando tablas en SQLite...")
  df_products.to_sql(
      "dim_products", engine, if_exists="replace", index=False
  )
  df_stores.to_sql("dim_stores", engine, if_exists="replace", index=False)
  df_oil.to_sql("fact_oil_prices", engine, if_exists="replace", index=False)

  # Para que la carga sea rapida, guardamos las ventas
  df_train.to_sql(
      "fact_daily_sales",
      engine,
      if_exists="replace",
      index=False,
      chunksize=50000,
  )

  print("Base de datos lista en data/retail_store.db!")


if __name__ == "__main__":
  build_database()

1/4 Cargando CSVs de Kaggle...
2/4 Creando tabla de dimensiones de productos/familias...
3/4 Preparando tabla de ventas y petroleo...
4/4 Guardando tablas en SQLite...
Base de datos lista en data/retail_store.db!


In [6]:
df_train = pd.read_csv("/content/drive/MyDrive/Data/train.csv")
df_train.info()
df_train.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0
6,6,2013-01-01,1,CELEBRATION,0.0,0
7,7,2013-01-01,1,CLEANING,0.0,0
8,8,2013-01-01,1,DAIRY,0.0,0
9,9,2013-01-01,1,DELI,0.0,0


In [9]:
# Crea un dataframe nuevo con la suma de sales por family
df_nuevo =df_train[["family","sales"]].groupby("family").sum().copy()
df_nuevo.head()




,sales
family,
AUTOMOTIVE,554822.0
BABY CARE,10051.0
BEAUTY,337893.0
BEVERAGES,216954486.0
BOOKS,6438.0


In [ ]:
df_stores = pd.read_csv("/content/drive/MyDrive/Data/stores.csv")
df_stores.info()
df_stores.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4
5,6,Quito,Pichincha,D,13
6,7,Quito,Pichincha,D,8
7,8,Quito,Pichincha,D,8
8,9,Quito,Pichincha,B,6
9,10,Quito,Pichincha,C,15


In [ ]:
df_oil = pd.read_csv("/content/drive/MyDrive/Data/oil.csv")
df_oil.info()
df_oil.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


In [ ]:
families = df_train["family"].unique()
np.random.seed(42)
df_products = pd.DataFrame({
      "product_id": families,
      "product_name": families,
      "category": "Retail",
      "unit_cost": np.random.uniform(2.5, 30.0, size=len(families)).round(2),
      "lead_time_days": np.random.choice([3, 5, 7, 10], size=len(families)),
  })
df_products["selling_price"] = (df_products["unit_cost"] * 1.30).round(2)
df_products.info()
df_products.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      33 non-null     object 
 1   product_name    33 non-null     object 
 2   category        33 non-null     object 
 3   unit_cost       33 non-null     float64
 4   lead_time_days  33 non-null     int64  
 5   selling_price   33 non-null     float64
dtypes: float64(2), int64(1), object(3)
memory usage: 1.7+ KB


,product_id,product_name,category,unit_cost,lead_time_days,selling_price
0,AUTOMOTIVE,AUTOMOTIVE,Retail,12.80,10,16.64
1,BABY CARE,BABY CARE,Retail,28.64,3,37.23
2,BEAUTY,BEAUTY,Retail,22.63,10,29.42
3,BEVERAGES,BEVERAGES,Retail,18.96,5,24.65
4,BOOKS,BOOKS,Retail,6.79,5,8.83
5,BREAD/BAKERY,BREAD/BAKERY,Retail,6.79,5,8.83
6,CELEBRATION,CELEBRATION,Retail,4.10,3,5.33
7,CLEANING,CLEANING,Retail,26.32,5,34.22
8,DAIRY,DAIRY,Retail,19.03,3,24.74
9,DELI,DELI,Retail,21.97,5,28.56


In [ ]:
#future

In [ ]:
import numpy as np
import pandas as pd


def prepare_time_series_features(df):
  """df debe tener columnas: ['date', 'family', 'sales', 'onpromotion']

  agrupado por día y familia (a nivel cadena o por tienda).
  """
  df = df.sort_values(["family", "date"]).reset_index(drop=True)

  # 1. Variables de Calendario
  df["dayofweek"] = df["date"].dt.dayofweek
  df["month"] = df["date"].dt.month
  df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

  # 2. Rezagos (Lags): ¿Qué pasó hace 7, 14, 28 días?
  lags = [7, 14, 21, 28, 30]
  for lag in lags:
    df[f"sales_lag_{lag}"] = df.groupby("family")["sales"].shift(lag)

  # 3. Ventanas móviles (Rolling): Promedio y dispersión histórica
  # Usamos shift(1) para evitar Data Leakage (fuga de datos del día actual)
  df["rolling_mean_7"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(7).mean()
  )
  df["rolling_std_7"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(7).std()
  )
  df["rolling_mean_30"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(30).mean()
  )

  # Eliminar los registros con NaN causados por los lags iniciales
  return df.dropna().reset_index(drop=True)


In [ ]:
#train and primize

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import os

# Creamos el archivo features.py de manera limpia
with open("features.py", "w") as f:
    f.write('''import numpy as np
import pandas as pd

def prepare_time_series_features(df):
  df = df.sort_values(["family", "date"]).reset_index(drop=True)
  df["dayofweek"] = df["date"].dt.dayofweek
  df["month"] = df["date"].dt.month
  df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

  lags = [7, 14, 21, 28, 30]
  for lag in lags:
    df[f"sales_lag_{lag}"] = df.groupby("family")["sales"].shift(lag)

  df["rolling_mean_7"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(7).mean()
  )
  df["rolling_std_7"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(7).std()
  )
  df["rolling_mean_30"] = df.groupby("family")["sales"].transform(
      lambda x: x.shift(1).rolling(30).mean()
  )
  return df.dropna().reset_index(drop=True)
''')


def train_inventory_model():
  # Asegurar existencia de carpetas para guardar resultados
  os.makedirs("data/processed", exist_ok=True)

  print("Cargando datos agregados por familia...")
  df = pd.read_csv("/content/drive/MyDrive/Data/train.csv")
  df["date"] = pd.to_datetime(df["date"])

  # Agregamos ventas a nivel nacional por familia para pronosticar demanda global
  daily_family_sales = (
      df.groupby(["date", "family"])
      .agg({"sales": "sum", "onpromotion": "sum"})
      .reset_index()
  )

  from features import prepare_time_series_features

  df_feat = prepare_time_series_features(daily_family_sales)

  # Split temporal: No usar train_test_split aleatorio
  # Entrenamos hasta junio 2017 y evaluamos en julio/agosto 2017
  train_mask = df_feat["date"] < "2017-07-01"
  test_mask = df_feat["date"] >= "2017-07-01"

  features = [
      col
      for col in df_feat.columns
      if col not in ["date", "sales", "family", "id"]
  ]
  # Convertir la columna 'family' a categorica para LightGBM
  df_feat["family"] = df_feat["family"].astype("category")
  features.append("family")

  X_train, y_train = df_feat[train_mask][features], df_feat[train_mask]["sales"]
  X_test, y_test = df_feat[test_mask][features], df_feat[test_mask]["sales"]

  print("Entrenando LightGBM...")
  model = lgb.LGBMRegressor(
      n_estimators=300,
      learning_rate=0.05,
      random_state=42,
      importance_type="gain",
  )
  model.fit(X_train, y_train)

  # Evaluacion
  preds = model.predict(X_test)
  rmse = np.sqrt(mean_squared_error(y_test, preds))
  print(f"RMSE en periodo de prueba: {rmse:.2f}")

  # ----------------------------------------------------
  # POLITICA DE INVENTARIO: Calculo de ROP y Stock de Seguridad
  # ----------------------------------------------------
  test_results = df_feat[test_mask].copy()
  test_results["prediction"] = preds
  test_results["error"] = test_results["sales"] - test_results["prediction"]

  # Nivel de servicio 95% -> Z = 1.65
  Z = 1.65
  LEAD_TIME_DAYS = 7  # Dias que tarda el proveedor

  inventory_kpis = []
  for fam in test_results["family"].cat.categories:
    sub = test_results[test_results["family"] == fam]
    if len(sub) == 0:
      continue

    sigma_error = sub["error"].std()
    avg_predicted_demand = sub["prediction"].mean()

    # Formulas de Supply Chain
    safety_stock = Z * sigma_error * np.sqrt(LEAD_TIME_DAYS)
    rop = (avg_predicted_demand * LEAD_TIME_DAYS) + safety_stock

    inventory_kpis.append({
        "family": fam,
        "daily_demand_pred": round(avg_predicted_demand, 2),
        "safety_stock": round(max(0, safety_stock), 2),
        "reorder_point (ROP)": round(max(0, rop), 2),
    })

  df_policy = pd.DataFrame(inventory_kpis)
  print("\n--- POLITICAS DE REORDEN CALCULADAS (Top 5) ---")
  print(df_policy.head())
  df_policy.to_csv("data/processed/inventory_policy.csv", index=False)


if __name__ == "__main__":
  train_inventory_model()

Cargando datos agregados por familia...
Entrenando LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002511 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2351
[LightGBM] [Info] Number of data points in the train set: 53064, number of used features: 13
[LightGBM] [Info] Start training from score 19300.161712
RMSE en periodo de prueba: 7022.69

--- POLITICAS DE REORDEN CALCULADAS (Top 5) ---
       family  daily_demand_pred  safety_stock  reorder_point (ROP)
0  AUTOMOTIVE             327.80        267.54              2562.17
1   BABY CARE              63.46        227.72               671.97
2      BEAUTY             434.75       1382.33              4425.56
3   BEVERAGES          195580.59      90623.86           1459688.00
4       BOOKS              63.46        228.60               672.84
